# Week 4：数据处理与数据集划分

目标：用 Pandas 检查数据质量，并在训练模型前正确划分训练集（train）、验证集（validation）和测试集（test）。

本 Notebook 使用 scikit-learn 自带的二分类数据集，只是为了学习规范流程；之后项目阶段会换成教育领域公开数据。

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

## 1. 读取数据并建立 DataFrame

机器学习中每一行通常是一条样本（sample），每一列通常是一个特征（feature），`target` 是要预测的标签（label）。

In [2]:
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()

print('shape:', df.shape)
print('target names:', list(dataset.target_names))
df.head()

shape: (569, 31)
target names: ['malignant', 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## 2. 数据体检（EDA 的最小版本）

开始建模前，至少检查：

- 数据形状、列名和数据类型；
- 缺失值；
- 标签分布是否不平衡；
- 是否存在明显不合理的数据。

In [3]:
print('data types:\n', df.dtypes.value_counts())
print('\nmissing values:', int(df.isna().sum().sum()))
print('\ntarget distribution:\n', df['target'].value_counts())
print('\ntarget ratio:\n', df['target'].value_counts(normalize=True).round(3))

data types:
 float64    30
int64       1
Name: count, dtype: int64

missing values: 0

target distribution:
 target
1    357
0    212
Name: count, dtype: int64

target ratio:
 target
1    0.627
0    0.373
Name: proportion, dtype: float64


## 3. 为什么要分训练、验证、测试集

$$\text{训练集}\rightarrow\text{学习参数}$$

$$\text{验证集}\rightarrow\text{选择模型和超参数}$$

$$\text{测试集}\rightarrow\text{最后一次客观评估}$$

测试集不能参与调参；否则模型会间接“记住”测试集，得到虚高结果，这叫 data leakage（数据泄漏）。

In [4]:
X = df.drop(columns='target')
y = df['target']

# 先分出 20% 测试集；stratify=y 保持各集合的标签比例接近。
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 剩余 80% 中再取 25% 作为验证集，即总数据的 20%。
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print('train:', X_train.shape, 'target ratio:', round(y_train.mean(), 3))
print('validation:', X_val.shape, 'target ratio:', round(y_val.mean(), 3))
print('test:', X_test.shape, 'target ratio:', round(y_test.mean(), 3))

train: (341, 30) target ratio: 0.628
validation: (114, 30) target ratio: 0.623
test: (114, 30) target ratio: 0.632


## 今日检查点

运行所有单元后，用自己的话回答：

1. `df.shape` 的两个数字分别表示什么？
2. 为什么这里需要 `stratify=y`？
3. 为什么不能用测试集来选择学习率或模型？

## 4. 预处理中的数据泄漏

标准化并不是简单地让所有数据都除以一个均值和标准差。正确顺序必须是：

$$X_{train}\xrightarrow{\mathrm{fit}}(\mu_{train},\sigma_{train})$$

$$X_{train},X_{val},X_{test}\xrightarrow{\mathrm{transform}}\frac{X-\mu_{train}}{\sigma_{train}}$$

若先用全部数据计算均值和标准差，测试集的分布信息就提前进入了训练过程，属于 data leakage。

同一原则也适用于缺失值填补、类别编码、特征选择和 PCA：所有会“学习统计量”的预处理器都只能在训练集上 `fit`。

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 只在训练集上学习每个特征的均值和标准差。
X_train_scaled = scaler.fit_transform(X_train)

# 验证集和测试集只能复用训练集的统计量。
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print('train mean (first 3):', X_train_scaled.iloc[:, :3].mean().round(3).to_dict())
print('train std (first 3):', X_train_scaled.iloc[:, :3].std(ddof=0).round(3).to_dict())
print('test mean (first 3):', X_test_scaled.iloc[:, :3].mean().round(3).to_dict())

train mean (first 3): {'mean radius': 0.0, 'mean texture': 0.0, 'mean perimeter': 0.0}
train std (first 3): {'mean radius': 1.0, 'mean texture': 1.0, 'mean perimeter': 1.0}
test mean (first 3): {'mean radius': 0.095, 'mean texture': 0.061, 'mean perimeter': 0.09}


### 检查点

为什么训练集标准化后均值接近 0、标准差接近 1，而测试集通常不会严格等于 0 和 1？请用自己的话回答。

## 5. 缺失值处理（missing values）

数值特征的常见做法是用中位数（median）填补。中位数比均值更不容易受极端值影响。关键原则不变：

$$X_{train}\xrightarrow{\mathrm{fit}}\text{每列的中位数}$$

$$X_{train},X_{val},X_{test}\xrightarrow{\mathrm{transform}}\text{用训练集中位数填补}$$

下面用一个小示例演示；当前主数据集没有缺失值，因此专门构造了带缺失值的训练集和验证集。

In [6]:
from sklearn.impute import SimpleImputer

train_demo = pd.DataFrame({
    'attendance': [0.9, 0.7, np.nan, 0.8],
    'score': [90, np.nan, 65, 75],
})

val_demo = pd.DataFrame({
    'attendance': [np.nan, 0.6],
    'score': [88, np.nan],
})

imputer = SimpleImputer(strategy='median')
train_filled = pd.DataFrame(
    imputer.fit_transform(train_demo), columns=train_demo.columns
)
val_filled = pd.DataFrame(
    imputer.transform(val_demo), columns=val_demo.columns
)

print('medians learned from train:', dict(zip(train_demo.columns, imputer.statistics_)))
print('\ntrain after imputation:\n', train_filled)
print('\nvalidation after imputation:\n', val_filled)

medians learned from train: {'attendance': 0.8, 'score': 75.0}

train after imputation:
    attendance  score
0         0.9   90.0
1         0.7   75.0
2         0.8   65.0
3         0.8   75.0

validation after imputation:
    attendance  score
0         0.8   88.0
1         0.6   75.0


### 检查点

验证集 `val_demo` 中的缺失值最终被填成什么数？这些数为什么不能由 `val_demo` 自己计算？

## 6. 类别特征编码（categorical encoding）

例如专业可能是 `CS`、`Math`、`English`。不能随意映射为 `CS=0`、`Math=1`、`English=2`，因为这会让线性模型误以为类别之间存在 $2>1>0$ 的数值顺序和距离。

one-hot encoding 为每个类别创建一列：

| major | major_CS | major_Math | major_English |
| --- | ---: | ---: | ---: |
| CS | 1 | 0 | 0 |
| Math | 0 | 1 | 0 |
| English | 0 | 0 | 1 |

同样遵循防泄漏原则：编码器只能在训练集 `fit`，验证集和测试集只能 `transform`。

In [7]:
from sklearn.preprocessing import OneHotEncoder

train_cat = pd.DataFrame({
    'major': ['CS', 'Math', 'CS', 'English'],
    'degree': ['Bachelor', 'Master', 'Bachelor', 'Master'],
})

# Physics 在训练集中没有出现，用于演示未知类别的处理。
val_cat = pd.DataFrame({
    'major': ['Math', 'Physics'],
    'degree': ['Master', 'Bachelor'],
})

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_encoded = encoder.fit_transform(train_cat)
val_encoded = encoder.transform(val_cat)

feature_names = encoder.get_feature_names_out()
train_encoded = pd.DataFrame(train_encoded, columns=feature_names)
val_encoded = pd.DataFrame(val_encoded, columns=feature_names)

print('encoded training data:\n', train_encoded)
print('\nencoded validation data:\n', val_encoded)

encoded training data:
    major_CS  major_English  major_Math  degree_Bachelor  degree_Master
0       1.0            0.0         0.0              1.0            0.0
1       0.0            0.0         1.0              0.0            1.0
2       1.0            0.0         0.0              1.0            0.0
3       0.0            1.0         0.0              0.0            1.0

encoded validation data:
    major_CS  major_English  major_Math  degree_Bachelor  degree_Master
0       0.0            0.0         1.0              0.0            1.0
1       0.0            0.0         0.0              1.0            0.0


### 检查点

为什么验证集中从未见过的 `Physics` 没有让代码报错？它在 `major` 对应的 one-hot 列中会是什么样？

## 7. Pipeline：把预处理规则固定下来

真实数据通常同时包含数值列和类别列。`ColumnTransformer` 让不同列走不同预处理路径；`Pipeline` 让每条路径按固定顺序执行。

$$\text{数值列}\rightarrow\text{中位数填补}\rightarrow\text{标准化}$$

$$\text{类别列}\rightarrow\text{众数填补}\rightarrow\text{one-hot 编码}$$

之后只需在训练集调用一次 `fit_transform`，在验证集和测试集调用 `transform`。

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X_train_mixed = pd.DataFrame({
    'attendance': [0.9, 0.7, np.nan, 0.8],
    'score': [90, np.nan, 65, 75],
    'major': ['CS', 'Math', 'CS', np.nan],
})

X_val_mixed = pd.DataFrame({
    'attendance': [np.nan, 0.6],
    'score': [88, np.nan],
    'major': ['Math', 'Physics'],
})

numeric_features = ['attendance', 'score']
categorical_features = ['major']

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

X_train_ready = preprocessor.fit_transform(X_train_mixed)
X_val_ready = preprocessor.transform(X_val_mixed)

print('processed train shape:', X_train_ready.shape)
print('processed validation shape:', X_val_ready.shape)
print('\nprocessed train:\n', X_train_ready)
print('\nprocessed validation:\n', X_val_ready)

processed train shape: (4, 4)
processed validation shape: (2, 4)

processed train:
 [[ 1.41421356  1.54030809  1.          0.        ]
 [-1.41421356 -0.14002801  0.          1.        ]
 [ 0.         -1.26025208  1.          0.        ]
 [ 0.         -0.14002801  1.          0.        ]]

processed validation:
 [[ 0.          1.31626328  0.          1.        ]
 [-2.82842712 -0.14002801  0.          0.        ]]


### 检查点

这个 `preprocessor` 中，哪一行代码真正从训练集学习了中位数、标准差、类别列表？为什么验证集只能调用 `transform`？